# 2020 — remaining F1 collection

**Restart the primusEnv kernel, then Run All.** This notebook now computes a fresh local audit and targets missing sessions and legacy identity repairs only. It does not repeat completed data or cancelled sessions.

The audit cell itself makes no API calls. Downloads start only in the collection cell. Existing raw caches and immutable publications are preserved. Earlier execution outputs were backed up locally before this update.


In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'data/archive/store.py').is_file() and (p / 'primusEnv').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the F1 Primus AI project.')
expected = (ROOT / 'primusEnv/Scripts/python.exe').resolve()
if Path(sys.executable).resolve() != expected:
    raise RuntimeError(f'Select the primusEnv kernel: {expected}')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Python:', sys.executable)


## Collection settings
The defaults collect the full eligible year without editing. 2026 is capped at October 26, 2026 and actual current time. Future/unfinished sessions stay out of the run.


In [ ]:
YEAR = 2020
THROUGH_DATE = '2026-10-26'  # Inclusive America/Chicago date; also bounded by current UTC time.
ROUND_NUMBERS = None        # None = every round; optional set such as {1, 2}.
RUN_DOWNLOAD = True         # Run All starts this collection; the agent has not run it.
DRY_RUN = False             # True = offline configuration preview only.

SESSION_LIMIT = None        # None = all eligible sessions; optional positive test limit.
RETRY_PARTIAL = True       # True explicitly revisits previously partial coverage.
AUTHENTICATE_LOCALLY = False # Reuse SDK sign-in from 00_setup; True prompts locally again.

VERIFY_EXISTING_FILES = False  # True repeats all source/Parquet hash checks before resuming.


## Audit checkpoint — October 26, 2026

Local source/Parquet checks passed. Of 83 calendar slots, three did not run: **Styrian Practice 3 and Eifel Practices 1–2**.
They are excluded, leaving 80 held-session slots. See the official [Styrian](https://www.formula1.com/en/racing/2020/styria)
and [Eifel](https://www.formula1.com/en/racing/2020/germany) schedules.

At the checkpoint, 62 sessions were published (35,999 laps). Remaining initial collection:

| Event | Sessions to collect |
|---|---|
| Styrian GP | Practice 1 — failed because substitute drivers had the same missing-ID sentinel; normalizer corrected |
| British GP | Practice 1 — driver-entry uniqueness error from missing-ID sentinel; normalizer corrected |
| Turkish GP | Practice 3 — deferred by rate limit |
| Bahrain GP | Practice 1, Practice 2, Practice 3, Qualifying, Race |
| Sakhir GP | Practice 1, Practice 2, Practice 3, Qualifying, Race |
| Abu Dhabi GP | Practice 1, Practice 2, Practice 3, Qualifying, Race |

Five already-published sessions also need **identity reprocessing from retained cache**:
Hungarian FP1, 70th Anniversary FP1, Spanish FP1, Italian FP1 and Eifel FP3.
The old normalizer treated literal `nan` as a driver/team ID. The corrected version keeps unresolved identities scoped to the session and car number instead of merging different people.

Other partial sessions contain flagged parser-filled/missing-time samples. They are **not automatically redownloaded**:
another request may return the same historical gaps. Set `RETRY_PARTIAL=True` only if you deliberately want to revisit them.
The live tables below supersede this checkpoint as your downloads complete.

In [ ]:
from data.archive.notebook_audit import audit_year

OUTPUT = ROOT / '.local/f1_2025_2026'
REMAINING_SESSION_IDS = None
REPROCESS_SESSION_IDS = set()
audit = None
if (OUTPUT / 'f1.sqlite3').is_file():
    audit = audit_year(OUTPUT, YEAR, verify_files=VERIFY_EXISTING_FILES)
    if audit['integrity']['sqlite'] != 'ok' or audit['integrity']['foreign_key_errors']:
        raise RuntimeError('Archive integrity failed. Preserve the archive and inspect before collecting.')
    print('Current audit:', audit['counts'])
    print('File verification:', audit['integrity'])
    rows = audit['sessions']
    selected = [r for r in rows if ROUND_NUMBERS is None or r['round_number'] in ROUND_NUMBERS]
    REMAINING_SESSION_IDS = {r['session_id'] for r in selected
                             if r['action'] in ('download', 'reprocess_identity')
                             or (RETRY_PARTIAL and r['action'] == 'review_partial')}
    REPROCESS_SESSION_IDS = {r['session_id'] for r in selected if r['action'] == 'reprocess_identity'}
    columns = ['session_id', 'event', 'session', 'checkpoint_status', 'laps', 'detail']
    for label, actions in [('Missing / retry / identity repair', {'download', 'reprocess_identity'}),
                           ('Saved with partial coverage — review separately', {'review_partial'}),
                           ('Not held — excluded from collection', {'not_held'}),
                           ('Local files needing repair — excluded until inspected', {'repair_local'})]:
        group = [r for r in selected if r['action'] in actions]
        print(label, len(group))
        if group:
            display(pd.DataFrame(group)[columns])
    print('Targeted sessions this run:', sorted(REMAINING_SESSION_IDS))
    if not rows:
        REMAINING_SESSION_IDS = None
        print('No calendar for this year is saved yet; collection will load it and exclude known cancelled sessions.')
else:
    print('No local calendar yet: the collection cell will build the full eligible 2020 plan.')

## Run / resume
Leave this cell running; it prints progress and provider cooldowns. Interrupt the kernel to stop safely. Run All again to resume and verify saved work. Access denial stops the run; sign in locally again if needed, then resume. Partial/unavailable data remain labelled. The data and HTTP accounting are shared across notebooks.


In [ ]:
if RUN_DOWNLOAD and not DRY_RUN:
    if REMAINING_SESSION_IDS is not None and not REMAINING_SESSION_IDS:
        print('No missing sessions or identity repairs remain. Partial quality flags are listed separately above.')
    else:
        from data.archive.notebook_f1 import run_f1_year
        progress = run_f1_year(ROOT, YEAR, through=THROUGH_DATE, round_numbers=ROUND_NUMBERS,
                               session_limit=SESSION_LIMIT, retry_partial=RETRY_PARTIAL,
                               authenticate=AUTHENTICATE_LOCALLY, session_ids=REMAINING_SESSION_IDS,
                               reprocess_session_ids=REPROCESS_SESSION_IDS)
        records = progress.get('sessions', {})
        if records:
            display(pd.DataFrame.from_dict(records, orient='index'))
else:
    print('Offline audit only: no sign-in, requests or download worker.')

## Next
Open `99_status_validate.ipynb` for an archive overview and integrity checks. Keep local caches and credentials out of Git/Kaggle. Publication is a later, separate step. See [the run guide](../../docs/data-collection-2020-2026.md).
